# ISL Translation Engine 2.0 — INCLUDE-50 sign recognition (Colab)

**Goal:** teach a model to look at a short video of one Indian Sign Language word and say which of 50 words it is.
**Data:** INCLUDE (AI4Bharat / IIT Madras), 943 usable videos of the 50-word subset. This is *word* recognition, not sentence translation.

## How to use this notebook — on day 1 and every day after

| Cell | What it does | Day 1 | Later days |
|---|---|---|---|
| **1. Settings** | your choices (tick the rights box once) | seconds | seconds |
| **2. Setup** | connects Drive, loads the code, installs packages | ~3 min | ~3 min (a new Colab machine is empty every time) |
| **3. Prepare data** | downloads videos, checks them, extracts hand/body landmarks | **long** (see below) | **seconds** — it restores from Drive |
| **4. Train & compare** | trains models, prints a comparison table | ~15–45 min | only trains what is missing |
| **5. Results / export** | lists finished runs, exports a model | seconds | seconds |

**You can press "Run all" safely:** every stage records what it finished on Google Drive and skips it next time.
Videos are fetched again **only** if landmarks are missing from Drive — after day 1 they never are.
If Colab disconnects, just run the same cells again; unfinished work continues, finished work is kept.

**Day-1 time (estimates — not measured on Colab):** videos ≈ 13 GB are the first big step; landmark extraction runs on one CPU core and
may take an hour or more. Progress is copied to Drive every two minutes while it runs.

**Runtime:** choose *Runtime → Change runtime type → CPU* (no GPU needed). This is word-level only; nothing here pushes to GitHub or deploys.

### Cell 1 — Settings
Change values on the right, then run. **Tick `RIGHTS_REVIEWED` only after you have read the dataset terms at https://zenodo.org/records/4010759** (licence shown today; either CC-BY or CC BY-NC-SA is fine for a private non-commercial demo — not legal advice).

In [ ]:
#@title 1. Settings { display-mode: "form" }
RIGHTS_REVIEWED = False #@param {type:"boolean"}
SPLITS = "official" #@param ["official", "session", "official + session"]
SEEDS = "13,14,15" #@param {type:"string"}
EPOCHS = 150 #@param {type:"integer"}
ALSO_TRY_AUGMENTATION = True #@param {type:"boolean"}
ALLOW_MISSING_VIDEOS = 0 #@param {type:"integer"}

### Cell 2 — Setup (run at the start of every session)
**What's happening:** (1) connects Google Drive, which is the only thing that survives between days; (2) unpacks the project code from the **newest**
`ISL_Translation_Engine*.zip` in `MyDrive/ISL_Translation_Engine/`; (3) installs the exact package versions with `uv` (it brings its own Python 3.12, so Colab's
Python 3.13 doesn't matter); (4) creates the workflow object `ctx`. Warnings like "unauthenticated requests to the HF Hub" are harmless.

In [ ]:
import os, sys, glob, zipfile, importlib
os.environ["MPLBACKEND"] = "Agg"        # Colab's inline-plot backend is not installed in the project environment
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/ISL_Translation_Engine")
DRIVE.mkdir(parents=True, exist_ok=True)
zips = sorted(glob.glob(str(DRIVE / "ISL_Translation_Engine*.zip")), key=os.path.getmtime)
assert zips, "Upload ISL_Translation_Engine_2_0_updated.zip into MyDrive/ISL_Translation_Engine/ first"
print("code package:", zips[-1])
with zipfile.ZipFile(zips[-1]) as z: z.extractall("/content")
%cd /content/ISL
!pip -q install uv
!uv sync --python 3.12 --group dev --extra experiments --locked
!uv run python -c "import isl, torch, mediapipe; print('environment OK | torch', torch.__version__, '| mediapipe', mediapipe.__version__)"
sys.path.insert(0, "/content/ISL")
from isl import flow
importlib.reload(flow)
_splits = {"official": ("official",), "session": ("session",), "official + session": ("official", "session")}[SPLITS]
_variants = ("baseline", "aug7-warp0.3") if ALSO_TRY_AUGMENTATION else ("baseline",)
ctx = flow.Ctx(project=Path("/content/ISL"), drive=DRIVE, settings=flow.Settings(
    rights_reviewed=RIGHTS_REVIEWED, policies=_splits, variants=_variants,
    seeds=tuple(int(x) for x in SEEDS.split(",") if x.strip()), epochs=EPOCHS, allow_missing_videos=ALLOW_MISSING_VIDEOS))
print()
print(flow.show_state(ctx))

### Cell 3 — Prepare data
**What's happening, stage by stage** (each is skipped if already finished; the output says what it did):
1. **audit** — reads INCLUDE's small metadata table once and keeps the list of the 943 INCLUDE-50 videos (one video was listed under two labels; those rows are excluded).
2. **videos** — makes the videos available: from this machine, else from your Drive copy, else from Zenodo. First a *pre-flight* reads each archive's file list (about 1 MB each,
   no videos) to confirm the folder layout matches; if the server supports it, only the 943 wanted videos are fetched (~13 GB) instead of all 56.8 GB. Each batch is saved to Drive as it finishes.
3. **checks** — builds the video list with train/validation/test splits, fingerprints each video, opens every video to confirm it decodes, and looks for near-duplicates. A failure **stops here** and tells you which report to read.
4. **landmarks** — MediaPipe marks the body and hand points in every frame and stores them as small number tables (this is what the model learns from). Saved to Drive; **later days restore this in seconds and never need the videos again.**

In [ ]:
flow.prepare(ctx)

### Cell 4 — Train & compare
**What's happening:** for each setting you chose, trains the model (a 1-D convolution → bidirectional LSTM → attention network, ~0.88 M parameters) and stores results on Drive.
Runs that already finished are skipped, so adding a new seed later only trains that seed. The table compares variants: **a difference only counts if it is bigger than the seed-to-seed spread and the test-set interval**
(the test set is only ~191 videos, so one video ≈ 0.5 points). *Baseline* = plain training; *aug7-warp0.3* adds small random tilt and speed changes (an idea from the AI4Bharat reference code — **not yet shown to help**).

In [ ]:
flow.train_all(ctx)
print()
print(flow.format_summary(flow.summary(ctx)))

### Cell 5 — Results and export
Lists the finished runs. To export one for the app, set `RUN_NAME` to a folder name from the list (check its `experiment.json` and per-class results first), then run the export line.
Export checks that the exported model matches the original; it does not mean the model is good enough — only the numbers above say that.

In [ ]:
runs = sorted(p.name for p in (DRIVE / "runs" / "isolated").glob("include50-*") if (p / "experiment.json").is_file())
print("finished runs:"); [print("  ", r) for r in runs]
RUN_NAME = ""   # e.g. "include50-official-baseline-seed13"
if RUN_NAME:
    !uv run python -m isl.export_onnx --ckpt {DRIVE}/runs/isolated/{RUN_NAME}/best.pt --out {DRIVE}/checkpoints/isl-{RUN_NAME}.onnx
    !ls -la {DRIVE}/checkpoints

## If something goes wrong
* **A cell stops with a red error:** read the last lines; stages print "Run this cell again" when that is the fix. Progress is kept on Drive.
* **"Only N of 943 wanted videos were found" / layout error:** nothing was downloaded. Copy the pre-flight lines above the error and send them to me.
* **Validation failed:** open the report named in the message (also copied to `Drive/reports/`).
* **Start a stage over:** run `flow.reset(ctx, "landmarks:official")` (names: `audit`, `videos`, `checked:official`, `landmarks:official`, same with `session`), then Cell 3. Files are not deleted.
* **See what is saved:** `print(flow.show_state(ctx))`.
* **Optional peek at the archive layout (downloads nothing):** `!uv run python -m isl.download_include --inspect all --keep /content/isl-data/keep.txt`